# **Feature Engineering Notebook**

## Objectives

* Load the cleaned train, test, and inherited houses datasets.
* Create simple engineered features that may help the machine learning model.
* Encode categorical variables into numerical values.
* Investigate numerical skewness and decide whether transformations are needed.
* Save feature-engineered datasets for modelling and dashboard prediction.

## Inputs

* `outputs/datasets/cleaned/TrainSetCleaned.csv`
* `outputs/datasets/cleaned/TestSetCleaned.csv`
* `outputs/datasets/cleaned/InheritedHousesCleaned.csv`

## Outputs

* `outputs/datasets/featured/TrainSetFeatured.csv`
* `outputs/datasets/featured/TestSetFeatured.csv`
* `outputs/datasets/featured/InheritedHousesFeatured.csv`
* `outputs/ml_pipeline/predict_sale_price/v1/selected_features.pkl`
* `outputs/ml_pipeline/predict_sale_price/v1/feature_engineering_mappings.pkl`

## Additional Comments

* This notebook prepares data for the machine learning model.
* Model training and evaluation will be handled in the next notebook.
* The same feature engineering steps must later be applied to user inputs in the Streamlit dashboard.

---

# Set Working Directory

Ensure the working directory is set to the project root directory.

In [ ]:
import os
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "jupyter_notebooks":
    os.chdir(current_dir.parent)
    print("You set a new current directory")
else:
    print("Current directory is already the project root or another non-notebook directory")

Path.cwd()

---

# Import Packages

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

---

# Load Cleaned Datasets

The cleaned datasets contain the string `"None"` for structural absence values such as no basement or no garage. When loading the cleaned CSV files, `keep_default_na=False` is used so that pandas preserves `"None"` as a category rather than converting it back into a missing value.

In [ ]:
TrainSet = pd.read_csv(
    "outputs/datasets/cleaned/TrainSetCleaned.csv",
    keep_default_na=False
)

TestSet = pd.read_csv(
    "outputs/datasets/cleaned/TestSetCleaned.csv",
    keep_default_na=False
)

InheritedHouses = pd.read_csv(
    "outputs/datasets/cleaned/InheritedHousesCleaned.csv",
    keep_default_na=False
)

In [ ]:
print("Train missing values:", TrainSet.isna().sum().sum())
print("Test missing values:", TestSet.isna().sum().sum())
print("Inherited houses missing values:", InheritedHouses.isna().sum().sum())

In [ ]:
TrainSet.head()

In [ ]:
TrainSet.shape, TestSet.shape, InheritedHouses.shape

In [ ]:
TrainSet.dtypes

# Separate Features and Target

In [ ]:
target = "SalePrice"

X_train = TrainSet.drop(columns=[target])
y_train = TrainSet[target]

X_test = TestSet.drop(columns=[target])
y_test = TestSet[target]

X_inherited = InheritedHouses.copy()

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape, X_inherited.shape

# Create Age-Related Features

The sale price study found that `YearBuilt`, `GarageYrBlt`, and `YearRemodAdd` were associated with `SalePrice`.

Raw year values are useful, but age-based features can sometimes be easier for a model and a user to interpret. Since the dataset covers houses built up to 2010, this notebook uses 2010 as the reference year for simple age calculations.

In [ ]:
REFERENCE_YEAR = 2010

In [ ]:
def add_age_features(df):
    """
    Add simple age-related features using 2010 as the reference year.
    """
    df = df.copy()

    df["HouseAge"] = REFERENCE_YEAR - df["YearBuilt"]
    df["RemodAge"] = REFERENCE_YEAR - df["YearRemodAdd"]

    # GarageYrBlt was set to 0 where there was no garage.
    # For houses with no garage, use 0 for GarageAge.
    df["GarageAge"] = np.where(
        df["GarageYrBlt"] == 0,
        0,
        REFERENCE_YEAR - df["GarageYrBlt"]
    )

    return df

In [ ]:
X_train = add_age_features(X_train)
X_test = add_age_features(X_test)
X_inherited = add_age_features(X_inherited)

In [ ]:
X_train[["YearBuilt", "HouseAge", "YearRemodAdd", "RemodAge", "GarageYrBlt", "GarageAge"]].head()

The original year columns are retained for now. Feature selection later in the notebook will decide which features to keep for modelling.

# Encode Categorical Variables

Machine learning models require numerical input. The categorical variables in this dataset describe ordered quality, finish, or exposure levels, so ordinal encoding is used.

The mappings are manually defined from lower to higher quality/exposure/finish based on the dataset metadata.

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.to_list()
categorical_cols

In [ ]:
ordinal_mappings = {
    "KitchenQual": {
        "Po": 0,
        "Fa": 1,
        "TA": 2,
        "Gd": 3,
        "Ex": 4
    },
    "BsmtExposure": {
        "None": 0,
        "No": 1,
        "Mn": 2,
        "Av": 3,
        "Gd": 4
    },
    "BsmtFinType1": {
        "None": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6
    },
    "GarageFinish": {
        "None": 0,
        "Unf": 1,
        "RFn": 2,
        "Fin": 3
    }
}

Validate first, before encoding:

In [ ]:
for col in categorical_cols:
    print(f"{col}:")
    print("Train:", sorted(X_train[col].dropna().unique()))
    print("Test:", sorted(X_test[col].dropna().unique()))
    print("Inherited:", sorted(X_inherited[col].dropna().unique()))
    print("Missing values:")
    print("  Train:", X_train[col].isna().sum())
    print("  Test:", X_test[col].isna().sum())
    print("  Inherited:", X_inherited[col].isna().sum())
    print()

Encoding function:

In [ ]:
def apply_ordinal_encoding(df, mappings):
    """
    Apply ordinal mappings to categorical variables.
    """
    df = df.copy()

    for col, mapping in mappings.items():
        if col in df.columns:
            df[col] = df[col].map(mapping)

    return df

In [ ]:
X_train = apply_ordinal_encoding(X_train, ordinal_mappings)
X_test = apply_ordinal_encoding(X_test, ordinal_mappings)
X_inherited = apply_ordinal_encoding(X_inherited, ordinal_mappings)

Check for failed mappings:

In [ ]:
X_train[categorical_cols].isna().sum()

In [ ]:
X_test[categorical_cols].isna().sum()

In [ ]:
X_inherited[categorical_cols].isna().sum()

The mapping checks confirm whether any categorical values failed to encode. Any missing values after this step would indicate that a category was not included in the mapping dictionary.

# Investigate Numerical Skewness

Some numerical variables, especially area-based features, may be skewed. Skewness is inspected to consider whether numerical transformations are needed.

For this project, transformations are not automatically applied. The final model is expected to use tree-based algorithms, which can handle non-normal feature distributions better than linear models. Applying transformations without a clear benefit could add unnecessary complexity to the Streamlit prediction process.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.to_list()

In [ ]:
skewness = X_train[numeric_features].skew().sort_values(ascending=False)
skewness

In [ ]:
high_skew_features = skewness[skewness.abs() > 1].index.to_list()
high_skew_features

In [ ]:
plt.figure(figsize=(10, 6))
skewness.sort_values().plot(kind="barh")
plt.title("Feature Skewness")
plt.xlabel("Skewness")
plt.ylabel("Feature")
plt.show()

Several numerical variables are skewed. This is expected for housing area features, because many houses have `0` for features such as second floor area, porch area, or wood deck area.

No numerical transformations are applied in this notebook. This keeps the feature engineering process simpler and avoids creating extra transformation steps that would also need to be applied to user input in the Streamlit dashboard.

# Select Modelling Features

In [ ]:
drop_features = [
    "YearBuilt",
    "YearRemodAdd",
    "GarageYrBlt"
]

In [ ]:
X_train = X_train.drop(columns=drop_features)
X_test = X_test.drop(columns=drop_features)
X_inherited = X_inherited.drop(columns=drop_features)

In [ ]:
selected_features = X_train.columns.to_list()
selected_features

In [ ]:
len(selected_features)

Check alignment:

In [ ]:
X_test = X_test[selected_features]
X_inherited = X_inherited[selected_features]

In [ ]:
X_train.shape, X_test.shape, X_inherited.shape

In [ ]:
X_train.head()

The original year columns are removed after creating age-based equivalents. This keeps the feature set easier to interpret while retaining the information carried by the construction, remodelling, and garage build years.

# Recombine Features and Target

In [ ]:
TrainSetFeatured = X_train.copy()
TrainSetFeatured[target] = y_train.values

TestSetFeatured = X_test.copy()
TestSetFeatured[target] = y_test.values

InheritedHousesFeatured = X_inherited.copy()

In [ ]:
TrainSetFeatured.head()

In [ ]:
TrainSetFeatured.shape, TestSetFeatured.shape, InheritedHousesFeatured.shape

# Save Feature-Engineered Datasets and Mappings

In [ ]:
featured_data_dir = Path("outputs/datasets/featured")
featured_data_dir.mkdir(parents=True, exist_ok=True)

pipeline_dir = Path("outputs/ml_pipeline/predict_sale_price/v1")
pipeline_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
TrainSetFeatured.to_csv(featured_data_dir / "TrainSetFeatured.csv", index=False)
TestSetFeatured.to_csv(featured_data_dir / "TestSetFeatured.csv", index=False)
InheritedHousesFeatured.to_csv(featured_data_dir / "InheritedHousesFeatured.csv", index=False)

In [ ]:
joblib.dump(selected_features, pipeline_dir / "selected_features.pkl")
joblib.dump(ordinal_mappings, pipeline_dir / "feature_engineering_mappings.pkl")
joblib.dump({"reference_year": REFERENCE_YEAR, "drop_features": drop_features}, pipeline_dir / "feature_engineering_config.pkl")

# Final Validation

In [ ]:
print("Train missing values:", TrainSetFeatured.isna().sum().sum())
print("Test missing values:", TestSetFeatured.isna().sum().sum())
print("Inherited missing values:", InheritedHousesFeatured.isna().sum().sum())

In [ ]:
TrainSetFeatured.dtypes

In [ ]:
TrainSetFeatured.select_dtypes(include=["object"]).columns.to_list()

# Conclusions and Next Steps

This notebook prepared the cleaned housing datasets for machine learning.

The main feature engineering decisions were:

* created age-related features from `YearBuilt`, `YearRemodAdd`, and `GarageYrBlt`;
* ordinal-encoded categorical variables using mappings based on the dataset metadata;
* investigated numerical skewness;
* decided not to apply skewness transformations at this stage to avoid unnecessary complexity;
* removed original year columns after creating age-based equivalents;
* saved the selected feature list, encoding mappings, and feature-engineering configuration.

The feature-engineered train, test, and inherited houses datasets were saved to `outputs/datasets/featured`.

The next notebook will train, tune, evaluate, and save a regression model to predict `SalePrice`.